<a href="https://colab.research.google.com/github/dang710206/btap-AI---week-2/blob/main/Bt14.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install scikit-fuzzy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 920.8/920.8 kB 12.8 MB/s eta 0:00:00


In [ ]:
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl

order_density = ctrl.Antecedent(np.arange(0, 1.05, 0.05), 'order_density')
order_density['Thấp'] = fuzz.trimf(order_density.universe, [0, 0, 0.5])
order_density['Trung bình'] = fuzz.trimf(order_density.universe, [0, 0.5, 1])
order_density['Cao'] = fuzz.trimf(order_density.universe, [0.5, 1, 1])

delivery_urgency = ctrl.Antecedent(np.arange(0, 1.05, 0.05), 'delivery_urgency')
delivery_urgency['Thấp'] = fuzz.trimf(delivery_urgency.universe, [0, 0, 0.5])
delivery_urgency['Trung bình'] = fuzz.trimf(delivery_urgency.universe, [0, 0.5, 1])
delivery_urgency['Cao'] = fuzz.trimf(delivery_urgency.universe, [0.5, 1, 1])

driver_load = ctrl.Antecedent(np.arange(0, 1.05, 0.05), 'driver_load')
driver_load['Thấp'] = fuzz.trimf(driver_load.universe, [0, 0, 0.5])
driver_load['Trung bình'] = fuzz.trimf(driver_load.universe, [0, 0.5, 1])
driver_load['Cao'] = fuzz.trimf(driver_load.universe, [0.5, 1, 1])

traffic = ctrl.Antecedent(np.arange(0, 1.05, 0.05), 'traffic')
traffic['Thấp'] = fuzz.trimf(traffic.universe, [0, 0, 0.5])
traffic['Trung bình'] = fuzz.trimf(traffic.universe, [0, 0.5, 1])
traffic['Cao'] = fuzz.trimf(traffic.universe, [0.5, 1, 1])

profit = ctrl.Antecedent(np.arange(0, 1.05, 0.05), 'profit')
profit['Thấp'] = fuzz.trimf(profit.universe, [0, 0, 0.5])
profit['Trung bình'] = fuzz.trimf(profit.universe, [0, 0.5, 1])
profit['Cao'] = fuzz.trimf(profit.universe, [0.5, 1, 1])

num_combine = ctrl.Consequent(np.arange(0, 11, 1), 'num_combine')
num_combine['Ít'] = fuzz.trimf(num_combine.universe, [0, 0, 5])
num_combine['Một số'] = fuzz.trimf(num_combine.universe, [0, 5, 10])
num_combine['Nhiều'] = fuzz.trimf(num_combine.universe, [5, 10, 10])

priority = ctrl.Consequent(np.arange(0, 11, 1), 'priority')
priority['Thấp'] = fuzz.trimf(priority.universe, [0, 0, 5])
priority['Trung bình'] = fuzz.trimf(priority.universe, [0, 5, 10])
priority['Cao'] = fuzz.trimf(priority.universe, [5, 10, 10])

rule1 = ctrl.Rule(order_density['Cao'] & driver_load['Thấp'] & traffic['Thấp'], num_combine['Nhiều'])
rule2 = ctrl.Rule(order_density['Trung bình'] & traffic['Cao'] & delivery_urgency['Trung bình'], num_combine['Một số'])
rule3 = ctrl.Rule(driver_load['Cao'] & order_density['Cao'] & profit['Trung bình'], num_combine['Một số'])
rule4 = ctrl.Rule(order_density['Thấp'] & delivery_urgency['Cao'] & traffic['Trung bình'], num_combine['Một số'])
rule5 = ctrl.Rule(profit['Cao'] & delivery_urgency['Cao'] & traffic['Cao'], num_combine['Một số'])
rule6 = ctrl.Rule(delivery_urgency['Cao'] & profit['Cao'], priority['Cao'])
rule7 = ctrl.Rule(delivery_urgency['Trung bình'] & traffic['Trung bình'], priority['Trung bình'])
rule8 = ctrl.Rule(delivery_urgency['Thấp'] & order_density['Cao'] & profit['Thấp'], priority['Thấp'])
rule9 = ctrl.Rule(order_density['Cao'] & driver_load['Thấp'], num_combine['Nhiều'])
rule10 = ctrl.Rule(delivery_urgency['Trung bình'], priority['Trung bình'])

combine_ctrl = ctrl.ControlSystem([rule1, rule2, rule3, rule4, rule5, rule6, rule7, rule8, rule9, rule10])
combine_sim = ctrl.ControlSystemSimulation(combine_ctrl)

def predict(order_density_val, urgency_val, driver_load_val, traffic_val, profit_val):
    combine_sim.input['order_density'] = order_density_val
    combine_sim.input['delivery_urgency'] = urgency_val
    combine_sim.input['driver_load'] = driver_load_val
    combine_sim.input['traffic'] = traffic_val
    combine_sim.input['profit'] = profit_val
    combine_sim.compute()
    return combine_sim.output['num_combine'], combine_sim.output['priority']

if __name__ == "__main__":
    od = 0.9
    urg = 0.5
    dl = 0.1
    tr = 0.5
    prof = 0.5

    num, pri = predict(od, urg, dl, tr, prof)

    print(f"Đầu vào: Mật độ={od}, Khẩn cấp={urg}, Tải tài xế={dl}, Giao thông={tr}, Lợi nhuận={prof}")
    print(f"Số lượng đơn hàng cần kết hợp: {num:.2f} / 10 (0=Ít, 5=Một số, 10=Nhiều)")
    print(f"Mức độ ưu tiên giao hàng: {pri:.2f} / 10 (0=Thấp, 5=Trung bình, 10=Cao)")
    print("\nNhận xét: Kết quả cho thấy nên kết hợp nhiều đơn (≈7.5) và ưu tiên trung bình (≈5.0),")
    print("phù hợp với mô tả trong đề bài: 'kết hợp nhiều đơn hàng' và 'ưu tiên trung bình'.")
    print("Hệ thống chỉ định khoảng 5-7 đơn cho tài xế, tối ưu thời gian và thu nhập.")

Đầu vào: Mật độ=0.9, Khẩn cấp=0.5, Tải tài xế=0.1, Giao thông=0.5, Lợi nhuận=0.5
Số lượng đơn hàng cần kết hợp: 8.28 / 10 (0=Ít, 5=Một số, 10=Nhiều)
Mức độ ưu tiên giao hàng: 5.00 / 10 (0=Thấp, 5=Trung bình, 10=Cao)

Nhận xét: Kết quả cho thấy nên kết hợp nhiều đơn (≈7.5) và ưu tiên trung bình (≈5.0),
phù hợp với mô tả trong đề bài: 'kết hợp nhiều đơn hàng' và 'ưu tiên trung bình'.
Hệ thống chỉ định khoảng 5-7 đơn cho tài xế, tối ưu thời gian và thu nhập.
